# Laboratorio guiado: almacenamiento en disco y desempeño de I/O

> **Instrucción general:** trabaja este notebook en orden, ejecutando celda por celda.  
> A medida que avances, conserva tus resultados y completa las secciones de análisis con tus propias observaciones.

---

## Cómo usar este notebook

En este laboratorio encontrarás tres tipos de secciones:

- **Ejecuta esta celda**: contiene código listo para usar.
- **Analiza**: debes interpretar los resultados obtenidos.
- **Conclusión**: debes redactar una respuesta breve con tus hallazgos.

### Recomendación

Antes de ejecutar todo, revisa la sección de **configuración** para ajustar el tamaño del archivo y el número de lecturas aleatorias si tu entorno es lento.


# Laboratorio: Almacenamiento en disco y desempeño de I/O

Este notebook permite medir y comparar:

- acceso secuencial vs acceso aleatorio
- distintos tamaños de bloque
- resultados empíricos vs estimaciones teóricas

La práctica está pensada para ejecutarse en Google Colab o en un entorno local con Python 3.


## 1. Objetivos del notebook

Al finalizar esta práctica deberías poder:

1. Entender cómo afecta el patrón de acceso al rendimiento.
2. Medir tiempos de lectura secuencial y aleatoria.
3. Calcular throughput empírico.
4. Comparar mediciones con un modelo teórico simple de I/O.
5. Visualizar los resultados con gráficas automáticas.


In [ ]:
# Librerías necesarias
import os
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Entorno listo.')


## 2. Recordatorio teórico breve

Usaremos las siguientes ideas:

### Throughput

$$\text{Throughput} = \frac{\text{datos transferidos}}{\text{tiempo total}}$$

### Modelo simplificado de costo I/O

$$\text{TotalTime} = \text{AccessLatency} \times M + \frac{\text{DataSize}}{\text{ScanThroughput}}$$

donde:

- **AccessLatency**: latencia para llegar al bloque
- **M**: número de accesos no contiguos
- **DataSize**: tamaño total de datos a leer
- **ScanThroughput**: velocidad de lectura sostenida

Interpretación:

- en acceso **secuencial**, normalmente $M \approx 1$
- en acceso **aleatorio**, $M$ puede ser muy grande


## Punto de control 1 — Revisión conceptual

Antes de continuar, responde brevemente:

1. ¿Qué representa la latencia en este laboratorio?
2. ¿Qué representa el throughput?
3. ¿Por qué en acceso secuencial normalmente se asume que $M \approx 1$?
4. ¿Por qué en acceso aleatorio $M$ tiende a ser mayor?

### Respuestas del estudiante

- Respuesta 1:
- Respuesta 2:
- Respuesta 3:
- Respuesta 4:


## 3. Configuración del experimento

Puedes ajustar estos parámetros según el tiempo disponible y la capacidad del entorno.


In [ ]:
# ==============================
# CONFIGURACIÓN GENERAL
# ==============================

DATA_DIR = Path('io_lab_data')
DATA_DIR.mkdir(exist_ok=True)

FILE_PATH = DATA_DIR / 'dataset.bin'

# Tamaño del archivo a generar.
# En Colab se recomienda entre 128 y 512 MB.
FILE_SIZE_MB = 256

# Tamaños de bloque a comparar
BLOCK_SIZES = [4 * 1024, 16 * 1024, 64 * 1024, 256 * 1024]

# Número de lecturas aleatorias por tamaño de bloque
RANDOM_READS = 4000

# Semilla para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Archivo:', FILE_PATH)
print('Tamaño objetivo (MB):', FILE_SIZE_MB)
print('Bloques a probar (bytes):', BLOCK_SIZES)
print('Número de lecturas aleatorias:', RANDOM_READS)


## Punto de control 2 — Verificación de parámetros

Registra aquí la configuración usada en tu experimento.

- Tamaño del archivo:
- Tamaños de bloque evaluados:
- Número de lecturas aleatorias:
- Entorno de ejecución (Colab / local):


## 4. Crear el archivo de prueba

Este archivo simula datos almacenados en disco. Solo se crea si todavía no existe o si su tamaño no coincide con la configuración actual.


In [ ]:
def create_test_file(file_path: Path, size_mb: int, chunk_mb: int = 8) -> None:
    """Crea un archivo binario grande para pruebas de I/O.

    Parámetros:
        file_path: ruta del archivo a crear.
        size_mb: tamaño total del archivo en MB.
        chunk_mb: tamaño del bloque con el que se escribe el archivo.
    """
    target_size = size_mb * 1024 * 1024
    if file_path.exists() and file_path.stat().st_size == target_size:
        print('El archivo ya existe y tiene el tamaño esperado.')
        return

    print(f'Creando archivo de {size_mb} MB...')
    chunk_bytes = chunk_mb * 1024 * 1024
    remaining = target_size

    with open(file_path, 'wb') as f:
        while remaining > 0:
            current = min(chunk_bytes, remaining)
            f.write(os.urandom(current))
            remaining -= current

    print('Archivo creado correctamente.')


create_test_file(FILE_PATH, FILE_SIZE_MB)
print('Tamaño final del archivo (bytes):', FILE_PATH.stat().st_size)


## Analiza

Después de crear el archivo, responde:

1. ¿Qué papel cumple este archivo dentro del experimento?
2. ¿Por qué es útil trabajar con un archivo relativamente grande?
3. ¿Qué crees que ocurriría si el archivo fuera demasiado pequeño?


## 5. Funciones auxiliares de medición


In [ ]:
def format_bytes(num_bytes: int) -> str:
    """Convierte un tamaño en bytes a una representación legible."""
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024


def throughput_mib_s(total_bytes: int, elapsed_seconds: float) -> float:
    """Calcula throughput en MiB/s."""
    if elapsed_seconds <= 0:
        return float('inf')
    return (total_bytes / (1024 * 1024)) / elapsed_seconds


def sequential_read_measure(file_path: Path, block_size: int) -> dict:
    """Mide lectura secuencial del archivo completo."""
    total_bytes = 0
    start = time.perf_counter()
    with open(file_path, 'rb', buffering=0) as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            total_bytes += len(chunk)
    elapsed = time.perf_counter() - start
    return {
        'pattern': 'Secuencial',
        'block_size_bytes': block_size,
        'total_bytes': total_bytes,
        'operations': math.ceil(total_bytes / block_size),
        'elapsed_s': elapsed,
        'throughput_mib_s': throughput_mib_s(total_bytes, elapsed),
    }


def random_read_measure(file_path: Path, block_size: int, reads: int) -> dict:
    """Mide lectura aleatoria de bloques en posiciones dispersas."""
    file_size = file_path.stat().st_size
    max_offset = file_size - block_size
    if max_offset <= 0:
        raise ValueError('El archivo es más pequeño que el tamaño de bloque.')

    offsets = np.random.randint(0, max_offset + 1, size=reads)
    total_bytes = 0

    start = time.perf_counter()
    with open(file_path, 'rb', buffering=0) as f:
        for offset in offsets:
            f.seek(int(offset))
            chunk = f.read(block_size)
            total_bytes += len(chunk)
    elapsed = time.perf_counter() - start
    return {
        'pattern': 'Aleatorio',
        'block_size_bytes': block_size,
        'total_bytes': total_bytes,
        'operations': reads,
        'elapsed_s': elapsed,
        'throughput_mib_s': throughput_mib_s(total_bytes, elapsed),
    }


## 6. Ejecutar experimentos empíricos

Se ejecutará la medición para cada tamaño de bloque, comparando acceso secuencial y aleatorio.


In [ ]:
results = []

for block_size in BLOCK_SIZES:
    print(f'\nProbando bloque = {format_bytes(block_size)}')

    seq_result = sequential_read_measure(FILE_PATH, block_size)
    rnd_result = random_read_measure(FILE_PATH, block_size, RANDOM_READS)

    results.append(seq_result)
    results.append(rnd_result)

    print(f"Secuencial -> tiempo: {seq_result['elapsed_s']:.4f} s, throughput: {seq_result['throughput_mib_s']:.2f} MiB/s")
    print(f"Aleatorio  -> tiempo: {rnd_result['elapsed_s']:.4f} s, throughput: {rnd_result['throughput_mib_s']:.2f} MiB/s")

df_empirical = pd.DataFrame(results)
df_empirical['block_size_kib'] = df_empirical['block_size_bytes'] / 1024
df_empirical


## Análisis de resultados empíricos

Observa la tabla generada y responde:

1. ¿Cuál patrón de acceso fue más rápido para cada tamaño de bloque?
2. ¿El throughput cambió al aumentar el tamaño de bloque?
3. ¿En qué caso viste la mayor diferencia entre secuencial y aleatorio?

### Respuesta del estudiante

Escribe aquí tu análisis.


## 7. Parámetros para el modelo teórico

Aquí puedes ajustar si quieres aproximar el comportamiento esperado de un HDD o de un SSD.

Las unidades son:

- latencia en segundos
- throughput en bytes por segundo


In [ ]:
# ==============================
# PARÁMETROS TEÓRICOS
# ==============================

THEORY_DEVICE = {
    'name': 'SSD aproximado',
    'access_latency_s': 10e-6,          # 10 microsegundos
    'scan_throughput_bytes_s': 5 * (1024**3),  # 5 GiB/s aprox.
}

# Si quieres modelar un HDD, puedes descomentar esto:
# THEORY_DEVICE = {
#     'name': 'HDD aproximado',
#     'access_latency_s': 10e-3,             # 10 milisegundos
#     'scan_throughput_bytes_s': 100 * (1024**2),  # 100 MiB/s aprox.
# }

print(THEORY_DEVICE)


## Punto de control 3 — Modelo teórico elegido

Indica cuál dispositivo teórico usaste para comparar tus resultados:

- Dispositivo modelado:
- Latencia asumida:
- Throughput asumido:

Luego explica por qué ese modelo podría parecerse o no a tu entorno real.


In [ ]:
def theoretical_io_time(data_size_bytes: int, m_accesses: int, access_latency_s: float, scan_throughput_bytes_s: float) -> float:
    """Aplica el modelo simplificado de costo I/O."""
    return access_latency_s * m_accesses + (data_size_bytes / scan_throughput_bytes_s)


theory_rows = []
file_size = FILE_PATH.stat().st_size

for block_size in BLOCK_SIZES:
    # Secuencial: idealizamos M = 1 para lectura contigua grande
    seq_data_size = file_size
    seq_m = 1
    seq_time = theoretical_io_time(
        data_size_bytes=seq_data_size,
        m_accesses=seq_m,
        access_latency_s=THEORY_DEVICE['access_latency_s'],
        scan_throughput_bytes_s=THEORY_DEVICE['scan_throughput_bytes_s'],
    )

    theory_rows.append({
        'pattern': 'Secuencial',
        'block_size_bytes': block_size,
        'block_size_kib': block_size / 1024,
        'theoretical_elapsed_s': seq_time,
        'theoretical_throughput_mib_s': throughput_mib_s(seq_data_size, seq_time),
    })

    # Aleatorio: M = número de accesos aleatorios
    rnd_data_size = block_size * RANDOM_READS
    rnd_m = RANDOM_READS
    rnd_time = theoretical_io_time(
        data_size_bytes=rnd_data_size,
        m_accesses=rnd_m,
        access_latency_s=THEORY_DEVICE['access_latency_s'],
        scan_throughput_bytes_s=THEORY_DEVICE['scan_throughput_bytes_s'],
    )

    theory_rows.append({
        'pattern': 'Aleatorio',
        'block_size_bytes': block_size,
        'block_size_kib': block_size / 1024,
        'theoretical_elapsed_s': rnd_time,
        'theoretical_throughput_mib_s': throughput_mib_s(rnd_data_size, rnd_time),
    })

df_theory = pd.DataFrame(theory_rows)
df_theory


## 8. Comparación entre resultados empíricos y teóricos


In [ ]:
df_compare = df_empirical.merge(
    df_theory,
    on=['pattern', 'block_size_bytes', 'block_size_kib'],
    how='left'
)

df_compare['elapsed_ratio_empirical_vs_theoretical'] = (
    df_compare['elapsed_s'] / df_compare['theoretical_elapsed_s']
)

df_compare['throughput_ratio_empirical_vs_theoretical'] = (
    df_compare['throughput_mib_s'] / df_compare['theoretical_throughput_mib_s']
)

df_compare


## Análisis comparativo: teoría vs práctica

Ahora interpreta la tabla comparativa:

1. ¿Los tiempos empíricos son mayores o menores que los teóricos?
2. ¿En cuál patrón de acceso la teoría se aproxima mejor?
3. ¿Qué factores reales podrían explicar las diferencias?

### Respuesta del estudiante

Escribe aquí tu comparación.


## 9. Gráficas automáticas

Las siguientes gráficas se generan automáticamente a partir de los resultados del experimento.


In [ ]:
def plot_empirical_throughput(df: pd.DataFrame) -> None:
    """Grafica throughput empírico para secuencial y aleatorio."""
    pivot = df.pivot(index='block_size_kib', columns='pattern', values='throughput_mib_s').sort_index()
    ax = pivot.plot(kind='bar', figsize=(10, 5))
    ax.set_title('Throughput empírico por tamaño de bloque')
    ax.set_xlabel('Tamaño de bloque (KiB)')
    ax.set_ylabel('Throughput (MiB/s)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_empirical_throughput(df_empirical)


## Interpreta la gráfica de throughput

Describe con tus palabras qué muestra esta gráfica.

- ¿Qué barras son más altas?
- ¿Qué significa eso en términos de rendimiento?
- ¿Cuál patrón aprovecha mejor la lectura en bloques?


In [ ]:
def plot_empirical_time(df: pd.DataFrame) -> None:
    """Grafica tiempo empírico por patrón de acceso."""
    pivot = df.pivot(index='block_size_kib', columns='pattern', values='elapsed_s').sort_index()
    ax = pivot.plot(marker='o', figsize=(10, 5))
    ax.set_title('Tiempo empírico por tamaño de bloque')
    ax.set_xlabel('Tamaño de bloque (KiB)')
    ax.set_ylabel('Tiempo (s)')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_empirical_time(df_empirical)


## Interpreta la gráfica de tiempo

Explica cómo cambia el tiempo total cuando cambia el tamaño de bloque.

### Respuesta del estudiante

Escribe aquí tu interpretación.


In [ ]:
def plot_theory_vs_empirical(df: pd.DataFrame, pattern: str) -> None:
    """Compara tiempo empírico y teórico para un patrón dado."""
    subset = df[df['pattern'] == pattern].sort_values('block_size_kib')
    plt.figure(figsize=(10, 5))
    plt.plot(subset['block_size_kib'], subset['elapsed_s'], marker='o', label='Empírico')
    plt.plot(subset['block_size_kib'], subset['theoretical_elapsed_s'], marker='o', label='Teórico')
    plt.title(f'Comparación tiempo empírico vs teórico - {pattern}')
    plt.xlabel('Tamaño de bloque (KiB)')
    plt.ylabel('Tiempo (s)')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_theory_vs_empirical(df_compare, 'Secuencial')
plot_theory_vs_empirical(df_compare, 'Aleatorio')


## Interpreta la comparación empírico vs teórico

Observa las curvas y responde:

1. ¿Las curvas tienen una tendencia similar?
2. ¿Dónde se separan más?
3. ¿Qué te sugiere eso sobre el modelo usado?


In [ ]:
def plot_sequential_speedup(df: pd.DataFrame) -> None:
    """Grafica cuántas veces el acceso secuencial supera al aleatorio."""
    pivot = df.pivot(index='block_size_kib', columns='pattern', values='throughput_mib_s').sort_index()
    speedup = pivot['Secuencial'] / pivot['Aleatorio']
    ax = speedup.plot(marker='o', figsize=(10, 5))
    ax.set_title('Ventaja del acceso secuencial sobre el aleatorio')
    ax.set_xlabel('Tamaño de bloque (KiB)')
    ax.set_ylabel('Factor de mejora (x)')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_sequential_speedup(df_empirical)


## Interpreta la ventaja del acceso secuencial

La última gráfica muestra cuántas veces el acceso secuencial supera al aleatorio.

- ¿Cuál fue el mayor factor de mejora observado?
- ¿Cómo cambia esa ventaja con el tamaño de bloque?
- ¿Qué implicación tiene esto para el diseño de estructuras de datos en disco?


## 10. Resumen automático de resultados


In [ ]:
summary_rows = []
for block_size in sorted(df_empirical['block_size_bytes'].unique()):
    seq = df_empirical[(df_empirical['block_size_bytes'] == block_size) & (df_empirical['pattern'] == 'Secuencial')].iloc[0]
    rnd = df_empirical[(df_empirical['block_size_bytes'] == block_size) & (df_empirical['pattern'] == 'Aleatorio')].iloc[0]
    summary_rows.append({
        'block_size_kib': block_size / 1024,
        'seq_time_s': seq['elapsed_s'],
        'rnd_time_s': rnd['elapsed_s'],
        'seq_throughput_mib_s': seq['throughput_mib_s'],
        'rnd_throughput_mib_s': rnd['throughput_mib_s'],
        'seq_vs_rnd_speedup': seq['throughput_mib_s'] / rnd['throughput_mib_s'] if rnd['throughput_mib_s'] > 0 else np.nan,
    })

df_summary = pd.DataFrame(summary_rows)
df_summary


## Conclusión final del laboratorio

Redacta una conclusión de **8 a 12 líneas** donde integres:

- cómo se almacena la información en bloques
- por qué el acceso secuencial y el aleatorio tienen desempeños distintos
- qué aprendiste al comparar teoría y práctica
- qué consecuencias tiene esto en el diseño de software y estructuras de datos

### Conclusión del estudiante

Escribe aquí tu conclusión final.


## 11. Preguntas para el análisis

Responde con base en los resultados obtenidos:

1. ¿Qué patrón de acceso obtuvo mayor throughput?
2. ¿Cómo cambió el rendimiento al variar el tamaño de bloque?
3. ¿En qué casos el modelo teórico se aproxima mejor al comportamiento observado?
4. ¿Qué factores reales pueden explicar las diferencias entre teoría y práctica?
5. ¿Qué implicaciones tiene esto para el diseño de estructuras de datos y sistemas de almacenamiento?


## 12. Extensiones sugeridas

Puedes ampliar el experimento de varias formas:

- repetir el experimento varias veces y promediar
- comparar lectura y escritura
- medir sobre SSD local vs disco de red
- cambiar el tamaño del archivo
- comparar caché caliente vs caché fría
